# Lớp tham chiếu cho công cụ lấy mẫu đa năm, đẩy lên Hugging Face

Dựng các ảnh để **đối chiếu khi gán nhãn** trong `LAY_MAU_DA_NAM.html`, rồi đẩy chúng lên một bộ dữ liệu Hugging Face công khai. Trang HTML đọc thẳng từ đó theo từng đoạn (COG), không phải tải cả tệp.

| lớp | nội dung | cỡ ước tính |
|---|---|---|
| `s2tc` | Sentinel-2 màu thật, ảnh tổng hợp mùa khô của luận án, 10 m, mỗi năm | ~300-400 MB/năm |
| `s2sw` | Sentinel-2 B11-B8-B4 (xây dựng tím hồng, thực vật xanh lá, nước đen) | ~300-400 MB/năm |
| `pca` | PCA chuỗi năm PC1-PC3, một cách kéo giãn chung cho mọi năm | ~300 MB/năm |
| `lulc_ctx` | bản đồ 3 lớp CTX đã làm mịn | ~15 MB/năm |
| `lulc_ctxts` | bản đồ 3 lớp CTX+TS11 của E6 (nếu E6 đã chạy xong) | ~15 MB/năm |
| `emb` (tuỳ chọn) | embedding dil_AB, 3 thành phần chính | ~300 MB/năm |
| `pc` (tuỳ chọn, NẶNG) | 11 PC int16 để trang tự tái dựng đường mùa vụ ở điểm BẤT KỲ | ~2-3 GB/năm |
| `diem/` | 900 điểm E0 (kèm nhãn 3 lớp cũ làm gợi ý) và điểm ứng viên phân tầng, **kèm đường NDVI, MNDWI 6 kỳ tái dựng từ PC cho mọi năm** | vài MB |

Mặc định không đẩy `pc`: đường mùa vụ đã được tính sẵn cho mọi điểm của hai bộ điểm, chỉ điểm thêm tay mới cần `pc`.

**Bộ dữ liệu mới, tách khỏi `lopmaybay/lulcHaiPhongCity`** (repo đó đã có DOI, không nên thêm 10 GB ảnh vào). Chạy trên runtime CPU là đủ (không cần GPU, trừ khi bật `emb` với ảnh embedding chưa có). Làm trên Drive nên chạy tiếp được nếu bị ngắt.

In [ ]:
# ============ 0. THAM SỐ ============
HF_REPO   = "lopmaybay/haiphong-lop-tham-chieu"   # bộ dữ liệu MỚI; đổi tên nếu muốn
CONG_KHAI = True        # trang HTML đọc thẳng từ trình duyệt: bộ dữ liệu phải công khai
NAM       = list(range(2017, 2027))
LAM = dict(s2tc=True, s2sw=True, pca=True, lulc_ctx=True, lulc_ctxts=True, emb=False, pc=False)
K_PC      = 11          # số PC đã chốt ở notebook PCA
N_UNG_VIEN = 1500       # số điểm ứng viên phân tầng
GUI_LEN   = False       # True: đẩy lên Hugging Face ở ô cuối
TAO_SPACE = False       # True: tạo thêm một Space tĩnh chứa trang HTML (có địa chỉ https)

!pip install -q geopandas rasterio huggingface_hub
from google.colab import drive; drive.mount("/content/drive")
import os, sys, glob, json, re, time, shutil
import numpy as np, pandas as pd, geopandas as gpd, rasterio
D = "/content/drive/MyDrive"; MOD = f"{D}/HP_modules"
OUT3 = f"{D}/HP_3class_v1"; MY = f"{OUT3}/multiyear"; TH = f"{OUT3}/thesis_v1"
DRIVE_MY = f"{D}/HP_multiyear"; CTX_V2 = f"{MY}_v2"; KQ_TS = f"{D}/HP_PCA_TS_KETQUA/S2"
HF_DIR = f"{D}/HP_HF_LOP_THAM_CHIEU"; os.makedirs(HF_DIR, exist_ok=True)
TMP = "/content/tmp_hf"; SRC = "/content/src_hf"; os.makedirs(TMP, exist_ok=True); os.makedirs(SRC, exist_ok=True)
NEED = ["s2_hf_lop.py", "s2_multiyear.py"]
thieu = [m for m in NEED if not os.path.exists(f"{MOD}/{m}")]
if thieu:
    print("Tải lên:", thieu); from google.colab import files
    for k in files.upload(): shutil.copy(k, MOD)
if MOD not in sys.path: sys.path.insert(0, MOD)
for m in ["s2_hf_lop", "s2_multiyear"]: sys.modules.pop(m, None)
import s2_hf_lop as H, s2_multiyear as M
assert H.self_test(verbose=False) == 0, "s2_hf_lop tự kiểm tra KHÔNG đạt"
print("s2_hf_lop", H.__version__, "| thư mục:", HF_DIR)

In [ ]:
# ============ 1. NGUỒN, LƯỚI, CÁCH KÉO GIÃN ============
import hashlib
cho = (glob.glob(f"{D}/HP_PCA_TS_CHUNG/he_so_S2.json")
       + glob.glob("/content/drive/.shortcut-targets-by-id/*/HP_PCA_TS_CHUNG/he_so_S2.json"))
HS = json.load(open(cho[0])); MA = HS["ma"]
TS_BY = {y: f"{KQ_TS}/HP_PCA_S2_{y}_{MA}" + ("_tam" if y == 2026 else "") + ".tif" for y in NAM}
RAS = {y: f"{DRIVE_MY}/S2_HP_{y}.tif" for y in NAM}
RAS.update({y: f"{DRIVE_MY}/S2_HP_{y}_v2.tif" for y in (2017, 2018)})     # ảnh đã vá
CTX_SM = {y: f"{CTX_V2}/smooth/LULC3_HP_{y}_smooth.tif" for y in NAM}
E6_SM = {y: f"{TH}/E6_pca_ts/series_ctx_ts{K_PC}_{MA}/smooth/LULC3_HP_{y}_smooth.tif" for y in NAM}
for ten, d in [("S2", RAS), ("PCA", TS_BY), ("CTX làm mịn", CTX_SM), ("CTX+TS (E6)", E6_SM)]:
    print(f"  {ten:12s}: có {[y for y, p in d.items() if os.path.exists(p)]}")
if not all(os.path.exists(E6_SM[y]) for y in range(2017, 2026)):
    LAM["lulc_ctxts"] = False; print("  -> chưa có chuỗi CTX+TS của E6: bỏ lớp lulc_ctxts (chạy lại khi E6 xong)")

bnd, _ = M.read_boundary(f"{D}/TÀI LIỆU KHOÁ HỌC VIỄN THÁM/DỮ LIỆU/Hải Phòng (phường xã) - 34.zip")
TF, WID, HEI, BND = M.boundary_grid(bnd, res=10, pad_m=200)
khung = (TF.c, TF.f + HEI * TF.e, TF.c + WID * TF.a, TF.f)
GRID = H.grid_3857(khung, "EPSG:32648", res=10.0, pad_m=300)
print(f"lưới 3857: {GRID[1]} × {GRID[2]} điểm ảnh 10 m")


def may(p):
    # chép nguồn về máy trước khi đọc (đọc qua Drive chậm); tên duy nhất theo đường dẫn
    q = f"{SRC}/{hashlib.md5(p.encode()).hexdigest()[:8]}_{os.path.basename(p)}"
    if not (os.path.exists(q) and os.path.getsize(q) == os.path.getsize(p)):
        shutil.copyfile(p, q)
    return q


def bo(p):
    q = f"{SRC}/{hashlib.md5(p.encode()).hexdigest()[:8]}_{os.path.basename(p)}"
    if os.path.exists(q):
        os.remove(q)


# kéo giãn CỐ ĐỊNH, tính một lần trên năm tham chiếu, dùng cho MỌI năm (màu so được giữa các năm)
KG_TC = H.percentiles(may(RAS[2025]), [3, 2, 1])            # B4, B3, B2
KG_SW = H.percentiles(may(RAS[2025]), [9, 7, 3])            # B11, B8, B4
KG_PC = H.percentiles(may(TS_BY[2023]), [1, 2, 3], nodata=H.TS_NODATA)
print("kéo giãn màu thật", np.round(KG_TC, 0).tolist(), "| B11-B8-B4", np.round(KG_SW, 0).tolist(),
      "| PC1-3", np.round(KG_PC, 0).tolist())

In [ ]:
# ============ 2. HAI BỘ ĐIỂM ============
os.makedirs(f"{HF_DIR}/diem", exist_ok=True)
YEAR_COLS = {2017: "cls_2017", 2019: "cls_2019", 2021: "cls_2021", 2023: "cls_2023", 2025: "ref_cls"}
pts = gpd.read_file(f"{TH}/E0_diem_kiem_dinh.gpkg")
lab = pd.read_csv(f"{TH}/E0_diem_kiem_dinh_da_gan.csv")
e0 = H.e0_points(pts, lab, YEAR_COLS)
e0.to_csv(f"{HF_DIR}/diem/E0_900.csv", index=False)
p_uv = f"{HF_DIR}/diem/ung_vien.csv"
if os.path.exists(p_uv):
    uv = pd.read_csv(p_uv); print("đã có", p_uv)
else:
    uv = H.candidate_points(CTX_SM[2025], may(TS_BY[2025]), n_total=N_UNG_VIEN, k_clusters=10,
                            min_dist=250, k_pc=K_PC)
    uv.to_csv(p_uv, index=False)
print("E0:", len(e0), "| ứng viên theo lớp bản đồ 2025:",
      uv.groupby("lop3_ban_do").size().rename({1: "thực vật", 2: "nước", 3: "xây dựng"}).to_dict())
he_so = {k: HS[k] for k in ["mean", "std", "W", "bang", "he_so_nhan"]}
he_so["W"] = HS["W"][:K_PC]
BO_DIEM = {"E0_900": e0, "ung_vien": uv}
DUONG = {}
for ten in BO_DIEM:
    p = f"{HF_DIR}/diem/{ten}_duong_cong.json"
    DUONG[ten] = json.load(open(p)) if os.path.exists(p) else dict(
        id=list(BO_DIEM[ten]["id"]), ky=HS.get("ky", [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]]),
        k=K_PC, nam={})
print("đường mùa vụ đã có:", {t: sorted(d["nam"]) for t, d in DUONG.items()})

In [ ]:
# ============ 3. DỰNG CÁC LỚP + ĐƯỜNG MÙA VỤ, TỪNG NĂM (bỏ qua phần đã có: chạy tiếp được) ============
def dung(lop, y, fn):
    out = f"{HF_DIR}/{lop}/{lop}_{y}.tif"
    if os.path.exists(out):
        return out
    os.makedirs(os.path.dirname(out), exist_ok=True)
    loc = f"{TMP}/{lop}_{y}.tif"
    fn(loc)
    shutil.copyfile(loc, out); os.remove(loc)
    return out


t0 = time.time()
for y in NAM:
    print(f"== {y}")
    if LAM["s2tc"] and os.path.exists(RAS[y]):
        dung("s2tc", y, lambda o: H.rgb_cog(may(RAS[y]), [3, 2, 1], *KG_TC, o, GRID, gamma=1.2, tmp_dir=TMP))
    if LAM["s2sw"] and os.path.exists(RAS[y]):
        dung("s2sw", y, lambda o: H.rgb_cog(may(RAS[y]), [9, 7, 3], *KG_SW, o, GRID, tmp_dir=TMP))
    if LAM["pca"] and os.path.exists(TS_BY[y]):
        dung("pca", y, lambda o: H.rgb_cog(may(TS_BY[y]), [1, 2, 3], *KG_PC, o, GRID, nodata=H.TS_NODATA, tmp_dir=TMP))
    if LAM["lulc_ctx"] and os.path.exists(CTX_SM[y]):
        dung("lulc_ctx", y, lambda o: H.class_cog(CTX_SM[y], o, GRID, tmp_dir=TMP))
    if LAM["lulc_ctxts"] and os.path.exists(E6_SM[y]):
        dung("lulc_ctxts", y, lambda o: H.class_cog(E6_SM[y], o, GRID, tmp_dir=TMP))
    if LAM["pc"] and os.path.exists(TS_BY[y]):
        dung("pc", y, lambda o: H.pc_cog(may(TS_BY[y]), o, k=K_PC, tmp_dir=TMP))
    if os.path.exists(TS_BY[y]):                      # đường mùa vụ tại hai bộ điểm, ghi ngay
        for ten, d in BO_DIEM.items():
            if str(y) in DUONG[ten]["nam"]:
                continue
            DUONG[ten]["nam"].update(H.pc_curves(d.x_utm.values, d.y_utm.values, [y], {y: may(TS_BY[y])},
                                                 he_so, k=K_PC))
            json.dump(DUONG[ten], open(f"{HF_DIR}/diem/{ten}_duong_cong.json", "w"))
    for p in (RAS[y], TS_BY[y]):                      # giữ đĩa máy gọn
        if p not in (RAS[2025], TS_BY[2023], TS_BY[2025]):
            bo(p)
    print(f"   {(time.time() - t0) / 60:.0f} phút")
for lop in ["s2tc", "s2sw", "pca", "lulc_ctx", "lulc_ctxts", "pc"]:
    fs = glob.glob(f"{HF_DIR}/{lop}/*.tif")
    if fs:
        print(f"  {lop:10s}: {len(fs)} tệp, {sum(os.path.getsize(f) for f in fs) / 1e9:.2f} GB")

In [ ]:
# ============ 4. (TUỲ CHỌN) EMBEDDING RGB ============
if LAM["emb"]:
    # ảnh embedding dil_AB đã ghi ở đường hai bước (s2_thesis.series_embedding); 64 băng, rất lớn
    EMB = {int(m.group(1)): p for p in glob.glob(f"{TH}/emb_dil_AB/*.tif")
           for m in [re.search(r"(20\d\d)", os.path.basename(p))] if m}
    print("ảnh embedding có:", sorted(EMB))
    if EMB:
        P3, MU, LO, HI = H.embed_pca(may(EMB[max(EMB)]))       # một phép chiếu chung cho mọi năm
        for y in sorted(EMB):
            dung("emb", y, lambda o: H.embed_rgb_cog(may(EMB[y]), o, GRID, P3, MU, LO, HI, tmp_dir=TMP))
            bo(EMB[y])
else:
    print("bỏ qua lớp embedding (LAM['emb'] = False)")

In [ ]:
# ============ 5. MANIFEST, HỆ THỐNG LỚP, README, TRANG HTML ============
MAU3 = {"1": "#2e9d3a", "2": "#1f5fbf", "3": "#d7191c"}


def co(lop):
    return sorted(int(re.findall(r"(20\d\d)", os.path.basename(p))[0]) for p in glob.glob(f"{HF_DIR}/{lop}/{lop}_*.tif"))


layers = []
for lop, ten, kieu in [("s2tc", "S2 màu thật (tổng hợp mùa khô)", "rgb"),
                       ("s2sw", "S2 B11-B8-B4 (xây dựng hồng tím, thực vật xanh, nước đen)", "rgb"),
                       ("pca", "PCA chuỗi năm PC1-PC3 (màu = kiểu mùa vụ)", "rgb"),
                       ("emb", "Embedding dil_AB, 3 thành phần chính", "rgb"),
                       ("lulc_ctx", "Bản đồ 3 lớp CTX (làm mịn)", "lop"),
                       ("lulc_ctxts", f"Bản đồ 3 lớp CTX+TS{K_PC} (E6)", "lop")]:
    ys = co(lop)
    if ys:
        L = dict(id=lop, ten=ten, kieu=kieu, duong_dan=f"{lop}/{lop}_{{y}}.tif", nam=ys)
        if kieu == "lop":
            L["bang_mau"] = MAU3
            L["ten_lop"] = {"1": "thực vật", "2": "nước", "3": "xây dựng"}
        layers.append(L)
pc = None
if co("pc"):
    pc = dict(duong_dan="pc/pc_{y}.tif", nam=co("pc"), k=K_PC, crs="EPSG:32648", he_so=he_so)
man = H.make_manifest(
    HF_REPO, layers, NAM, grid_utm=dict(crs="EPSG:32648", x0=615610.0, y0=2352020.0, res=10.0),
    point_sets=[dict(id="E0", ten="900 điểm kiểm định E0 (gợi ý = nhãn 3 lớp cũ)",
                     duong_dan="diem/E0_900.csv", duong_cong="diem/E0_900_duong_cong.json"),
                dict(id="UV", ten=f"{len(uv)} điểm ứng viên phân tầng (lớp CTX 2025 × cụm PCA)",
                     duong_dan="diem/ung_vien.csv", duong_cong="diem/ung_vien_duong_cong.json")],
    curves=dict(feats=["NDVI", "MNDWI"], ky=HS.get("ky"), k=K_PC), pc=pc,
    ghi_chu=f"PCA mã {MA}, {K_PC} PC; 2017, 2018 ảnh S2 L2A thưa (xem bảng trôi); 2026 là năm chưa trọn.")
json.dump(man, open(f"{HF_DIR}/manifest.json", "w"), ensure_ascii=False, indent=1)
for f in ["he_lop_v1.json", "LAY_MAU_DA_NAM.html"]:
    if os.path.exists(f"{MOD}/{f}"):
        shutil.copyfile(f"{MOD}/{f}", f"{HF_DIR}/{f}")
    else:
        print(f"  (thiếu {f} trong HP_modules: tải lên đó nếu muốn đưa kèm)")
readme = f'''---
license: cc-by-4.0
tags: [remote-sensing, sentinel-2, land-cover, haiphong, vietnam, cog]
---
# Lớp tham chiếu cho lấy mẫu lớp phủ đa năm, Hải Phòng (2017-2026)

Ảnh GeoTIFF tối ưu đám mây (COG, EPSG:3857, 10 m) dùng để đối chiếu khi gán nhãn trong công cụ
`LAY_MAU_DA_NAM.html`: {", ".join(L["id"] for L in layers)}. Kèm hai bộ điểm và đường NDVI, MNDWI 6 kỳ
tái dựng từ PCA chuỗi thời gian ({K_PC} PC, hệ số chung cho mọi năm).

Tác giả: Phạm Đăng Hiển (MIIGAiK). Contains modified Copernicus Sentinel data (2017-2026), processed in
Google Earth Engine. Xem `manifest.json` để biết danh mục tệp.
'''
open(f"{HF_DIR}/README.md", "w", encoding="utf-8").write(readme)
tong = sum(os.path.getsize(p) for p in glob.glob(f"{HF_DIR}/**/*", recursive=True) if os.path.isfile(p))
print(f"manifest: {len(layers)} lớp, {len(man['point_sets'])} bộ điểm | tổng {tong / 1e9:.2f} GB trong {HF_DIR}")

In [ ]:
# ============ 6. ĐẨY LÊN HUGGING FACE ============
from huggingface_hub import HfApi, login
try:
    from google.colab import userdata
    TOKEN = userdata.get("HF_TOKEN")          # đặt ở Colab: biểu tượng chìa khoá (Secrets), tên HF_TOKEN
except Exception:
    from getpass import getpass
    TOKEN = getpass("Mã truy cập Hugging Face (quyền write): ")
api = HfApi(token=TOKEN)
if GUI_LEN:
    api.create_repo(HF_REPO, repo_type="dataset", private=not CONG_KHAI, exist_ok=True)
    api.upload_large_folder(folder_path=HF_DIR, repo_id=HF_REPO, repo_type="dataset",
                            ignore_patterns=["*.tmp", ".cache/*"])
    print(f"xong: https://huggingface.co/datasets/{HF_REPO}")
    print("thử manifest:", f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/manifest.json")
else:
    print("GUI_LEN = False: chưa đẩy. Đặt True ở ô 0 rồi chạy lại ô này.")
if TAO_SPACE and os.path.exists(f"{HF_DIR}/LAY_MAU_DA_NAM.html"):
    sp = HF_REPO.split("/")[0] + "/lay-mau-hai-phong"
    api.create_repo(sp, repo_type="space", space_sdk="static", private=False, exist_ok=True)
    api.upload_file(path_or_fileobj=f"{HF_DIR}/LAY_MAU_DA_NAM.html", path_in_repo="index.html",
                    repo_id=sp, repo_type="space")
    print(f"trang: https://huggingface.co/spaces/{sp}")